# RWKV-7 for Natural Language Inference

**COMP34812 NLU Coursework - Category B (Non-Transformer Architecture)**

Fine-tunes the RWKV-7 "Goose" (7.2B parameters) model for binary NLI classification using LoRA. Optimized for GPUs with BF16 mixed precision training.

## 1. Setup & Installation

Install required packages: PyTorch with CUDA support, Hugging Face transformers, PEFT for LoRA, and flash-linear-attention for RWKV-7. Uncomment and run if needed.

In [1]:
# %pip install --upgrade pip
#
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
#
# %pip install transformers peft accelerate scikit-learn pandas matplotlib tqdm
# %pip install triton
#
# # %pip uninstall -y torch torchvision torchaudio
# %pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128
#
# # NOTE: For RWKV7, the creators often recommend grabbing the absolute latest from GitHub:
# # %pip install git+https://github.com/fla-org/flash-linear-
# %pip install flash-linear-attention

In [2]:
# %pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128 --upgrade --force-reinstall

In [ ]:
# Imports
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from torch.optim import AdamW
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef, classification_report
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import gc

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

Define all hyperparameters and paths in a single dictionary for easy modification. Includes model settings, data paths, training parameters (batch size, learning rate, epochs), LoRA configuration, and hardware-specific options.

In [ ]:
# ============================================================
# CONFIGURATION - RTX 5090 Optimized
# ============================================================
CONFIG = {
    # Model - RWKV-7 "Goose"
    "model_name": "fla-hub/rwkv7-7.2B-g0a",
    "num_labels": 2,

    # Data paths
    "train_path": "Data/train.csv",
    "dev_path": "Data/dev.csv",
    "max_length": 320,

    # Training - RTX 5090 optimized
    "batch_size": 3,            # RTX 5090 32GB can handle this
    "grad_accum_steps": 6,      # Effective batch size: 18
    "learning_rate": 1e-5,
    "num_epochs": 5,
    "warmup_ratio": 0.1,
    "num_workers": 2,

    # LoRA
    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,

    # Other
    "dropout": 0.1,
    "seed": 42,
    "output_dir": "rwkv",
    "checkpoint_dir": "rwkv_checkpoints",
    "resume_from_checkpoint": None,  # Set to path to resume

    # RTX 5090 specific
    "use_bf16": True,           # BF16 is better on Blackwell
    "use_torch_compile": False,  # PyTorch 2.0+ for speed
}

# Set seed
torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

print("Configuration loaded!")

## 3. Device Setup

Detect available hardware (CUDA GPU or CPU), check BF16 support, and enable performance optimizations (TF32, cuDNN benchmark). Automatically falls back to FP16 or CPU if needed.

In [5]:
def get_device():
    """Get the best available device."""
    if torch.cuda.is_available():
        device = torch.device("cuda")
        props = torch.cuda.get_device_properties(0)
        print(f"Using CUDA: {props.name}")
        print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
        print(f"Compute Capability: {props.major}.{props.minor}")

        # Check BF16 support
        if torch.cuda.is_bf16_supported():
            print("BF16: Supported ✓")
        else:
            print("BF16: Not supported, using FP16")
            CONFIG["use_bf16"] = False

        # Enable TF32 for faster matmuls
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
        print("TF32 and cuDNN benchmark: Enabled")

        return device
    else:
        print("CUDA not available, using CPU")
        CONFIG["use_bf16"] = False
        CONFIG["use_torch_compile"] = False
        return torch.device("cpu")

device = get_device()

Using CUDA: NVIDIA GeForce RTX 5090
VRAM: 34.2 GB
Compute Capability: 12.0
BF16: Supported ✓
TF32 and cuDNN benchmark: Enabled


## 4. Dataset

PyTorch Dataset class for NLI data. Loads premise-hypothesis pairs from CSV, tokenizes them into the format `"Premise: {text}\nHypothesis: {text}"`, and returns input tensors with labels.

In [6]:
class NLIDataset(Dataset):
    """NLI Dataset for RWKV."""

    def __init__(self, csv_path, tokenizer, max_length=256, is_test=False):
        self.df = pd.read_csv(csv_path)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_test = is_test
        print(f"Loaded {len(self.df)} samples from {csv_path}")
        
        if not is_test and "label" in self.df.columns:
            print(f"Label distribution: {self.df['label'].value_counts().to_dict()}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = f"Premise: {row['premise']}\nHypothesis: {row['hypothesis']}"

        enc = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
        }

        if not self.is_test and "label" in row:
            item["labels"] = torch.tensor(int(row["label"]), dtype=torch.long)

        return item

## 5. Model Definition

`RWKV7ForNLI` wraps the pre-trained RWKV-7 backbone with a classification head. Uses last-token pooling to extract sequence representations, then passes through LayerNorm, Dropout, and a Linear layer for binary classification. Includes `save()` and `load()` methods for model persistence.

In [ ]:
class RWKV7ForNLI(nn.Module):
    """
    RWKV-7 "Goose" for NLI Classification.

    Category B: Non-transformer architecture
    - 100% RNN (no attention mechanism)
    - Linear time complexity O(n)
    - Constant space complexity O(1)
    - Uses delta rule for state updates
    """

    def __init__(self, model_name, num_labels=2, dropout=0.1, use_bf16=True):
        super().__init__()

        dtype = torch.bfloat16 if use_bf16 else torch.float16
        print(f"Loading {model_name} with {dtype}...")

        self.backbone = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            trust_remote_code=True,
        )

        self.hidden_size = self.backbone.config.hidden_size
        self.dtype = dtype
        print(f"Hidden size: {self.hidden_size}")

        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.hidden_size),
            nn.Dropout(dropout),
            nn.Linear(self.hidden_size, num_labels)
        )

        # Initialize
        for m in self.classifier:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.backbone(
            input_ids=input_ids,
            output_hidden_states=True,
            return_dict=True,
        )

        hidden = outputs.hidden_states[-1]

        # Pool: last non-padding token
        if attention_mask is not None:
            seq_lens = attention_mask.sum(dim=1) - 1
            seq_lens = seq_lens.clamp(min=0)
            batch_idx = torch.arange(hidden.size(0), device=hidden.device)
            pooled = hidden[batch_idx, seq_lens]
        else:
            pooled = hidden[:, -1, :]

        # Cast to float32 for classifier stability
        pooled = pooled.float()
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

    def save(self, path):
        os.makedirs(path, exist_ok=True)
        # Save the PEFT model (backbone with LoRA adapters)
        self.backbone.save_pretrained(os.path.join(path, "backbone"))
        torch.save(self.classifier.state_dict(), os.path.join(path, "classifier.pt"))
        torch.save({
            "hidden_size": self.hidden_size,
            "dtype": str(self.dtype)
        }, os.path.join(path, "config.pt"))
        print(f"Model saved to {path}")

    @classmethod
    def load(cls, path, device, use_bf16=True):
        """Load a saved model with LoRA adapters."""
        from peft import PeftModel
        
        config = torch.load(os.path.join(path, "config.pt"), weights_only=False)
        model = cls.__new__(cls)
        nn.Module.__init__(model)

        dtype = torch.bfloat16 if use_bf16 else torch.float16
        backbone_path = os.path.join(path, "backbone")
        
        # Check if this is a PEFT model by looking for adapter_config.json
        adapter_config_path = os.path.join(backbone_path, "adapter_config.json")
        
        if os.path.exists(adapter_config_path):
            # Load as PEFT model: first load base, then load adapters
            import json
            with open(adapter_config_path, 'r') as f:
                adapter_config = json.load(f)
            base_model_name = adapter_config.get("base_model_name_or_path", "fla-hub/rwkv7-7.2B-g0a")
            
            print(f"Loading base model: {base_model_name}")
            base_model = AutoModelForCausalLM.from_pretrained(
                base_model_name,
                torch_dtype=dtype,
                trust_remote_code=True,
            )
            
            print(f"Loading LoRA adapters from: {backbone_path}")
            model.backbone = PeftModel.from_pretrained(base_model, backbone_path)
        else:
            # Fallback: load as regular model (no LoRA)
            print(f"Loading model from: {backbone_path}")
            model.backbone = AutoModelForCausalLM.from_pretrained(
                backbone_path,
                torch_dtype=dtype,
                trust_remote_code=True,
            )
        
        model.hidden_size = config["hidden_size"]
        model.dtype = dtype

        model.classifier = nn.Sequential(
            nn.LayerNorm(model.hidden_size),
            nn.Dropout(0.1),
            nn.Linear(model.hidden_size, 2)
        )
        model.classifier.load_state_dict(
            torch.load(os.path.join(path, "classifier.pt"), weights_only=False)
        )

        print("Model loaded successfully!")
        return model.to(device)

## 6. Checkpoint Functions

Utility functions to save and load training checkpoints. Saves model weights, optimizer state, scheduler state, best metrics, and training history. Enables resuming training from any epoch.

In [8]:
def save_checkpoint(model, optimizer, scheduler, epoch, best_metrics, history, path):
    """Save training checkpoint."""
    os.makedirs(path, exist_ok=True)
    
    # Handle compiled model
    model_to_save = model._orig_mod if hasattr(model, "_orig_mod") else model
    model_to_save.save(os.path.join(path, "model"))
    
    torch.save({
        "epoch": epoch,
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_f1": best_metrics["f1"],
        "best_mcc": best_metrics["mcc"],
        "best_combined": best_metrics["combined"],
        "history": history,
    }, os.path.join(path, "training_state.pt"))
    print(f"Checkpoint saved to {path}")


def load_checkpoint(path, optimizer, scheduler, device):
    """Load training checkpoint."""
    model = RWKV7ForNLI.load(os.path.join(path, "model"), device, CONFIG["use_bf16"])
    state = torch.load(os.path.join(path, "training_state.pt"))
    optimizer.load_state_dict(state["optimizer_state_dict"])
    scheduler.load_state_dict(state["scheduler_state_dict"])
    return {
        "model": model,
        "epoch": state["epoch"],
        "best_metrics": {
            "f1": state["best_f1"],
            "mcc": state["best_mcc"],
            "combined": state["best_combined"],
        },
        "history": state["history"],
    }

## 7. Training Functions

- `train_epoch()`: Runs one training epoch with gradient accumulation, mixed precision (BF16/FP16), and gradient clipping.
- `evaluate()`: Computes loss, F1 score, accuracy, and MCC on the validation set without gradient computation.

In [9]:
def train_epoch(model, loader, optimizer, scheduler, device, accum_steps, use_bf16):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    autocast_dtype = torch.bfloat16 if use_bf16 else torch.float16
    pbar = tqdm(loader, desc="Training")
    
    for step, batch in enumerate(pbar):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(dtype=autocast_dtype):
            out = model(input_ids, attention_mask, labels)
            loss = out["loss"] / accum_steps

        loss.backward()

        if (step + 1) % accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps
        pbar.set_postfix({"loss": f"{loss.item()*accum_steps:.4f}"})

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device, use_bf16):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    autocast_dtype = torch.bfloat16 if use_bf16 else torch.float16

    for batch in tqdm(loader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(dtype=autocast_dtype):
            out = model(input_ids, attention_mask, labels)

        preds = torch.argmax(out["logits"], dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        total_loss += out["loss"].item()

    return {
        "loss": total_loss / len(loader),
        "f1": f1_score(all_labels, all_preds),
        "accuracy": accuracy_score(all_labels, all_preds),
        "mcc": matthews_corrcoef(all_labels, all_preds),
        "preds": all_preds,
        "labels": all_labels,
    }

## 8. Load Model & Data

Load the tokenizer, instantiate the RWKV-7 model, apply LoRA adapters to reduce trainable parameters, and create DataLoaders for training and validation sets.

In [10]:
# Create output directories
os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Vocab size: {tokenizer.vocab_size}")

Loading tokenizer...
Vocab size: 65531


In [11]:
# Load model
print("Loading model...")
model = RWKV7ForNLI(
    model_name=CONFIG["model_name"],
    num_labels=CONFIG["num_labels"],
    dropout=CONFIG["dropout"],
    use_bf16=CONFIG["use_bf16"]
)

# Apply LoRA
print("\nApplying LoRA...")
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=["key", "value", "receptance", "gate", "output"],
    bias="none",
)
model.backbone = get_peft_model(model.backbone, lora_config)
model.backbone.print_trainable_parameters()

# Move to device
model = model.to(device)

Loading model...
Loading fla-hub/rwkv7-2.9B-world with torch.bfloat16...


`torch_dtype` is deprecated! Use `dtype` instead!
/home/skynet/git/NLU/venv313/lib/python3.13/site-packages/fla/layers/rwkv7.py:143: UserWarning: According to Bo, you are using a potentially buggy FLA implementation of RWKV. If you plan to report any numbers based on this implementation, we strongly recommend cross-checking with the official repo: https://github.com/BlinkDL/RWKV-LM. Bo may disagree with results reported from this version.
  warnings.warn(


Loading weights:   0%|          | 0/1059 [00:00<?, ?it/s]

Hidden size: 2560

Applying LoRA...
trainable params: 26,214,400 || all params: 2,973,949,440 || trainable%: 0.8815


In [12]:
# Compile model for speed (PyTorch 2.0+)
if CONFIG["use_torch_compile"] and hasattr(torch, "compile"):
    print("Compiling model with torch.compile...")
    model = torch.compile(model, mode="reduce-overhead")
    print("Model compiled ✓")

In [13]:
# Load datasets
print("Loading datasets...")
train_dataset = NLIDataset(CONFIG["train_path"], tokenizer, CONFIG["max_length"])
dev_dataset = NLIDataset(CONFIG["dev_path"], tokenizer, CONFIG["max_length"])

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
    # persistent_workers=False,
    persistent_workers=True,
)
dev_loader = DataLoader(
    dev_dataset,
    batch_size=CONFIG["batch_size"] * 2,
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
    # persistent_workers=False,
    persistent_workers=True,
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Dev batches: {len(dev_loader)}")

Loading datasets...
Loaded 24432 samples from Data/train.csv
Label distribution: {1: 12648, 0: 11784}
Loaded 6736 samples from Data/dev.csv
Label distribution: {1: 3478, 0: 3258}

Train batches: 6108
Dev batches: 842


## 9. Setup Optimizer & Scheduler

Initialize AdamW optimizer with weight decay and a linear learning rate scheduler with warmup. Calculates total training steps based on epochs and gradient accumulation.

In [14]:
# Optimizer & Scheduler
steps_per_epoch = len(train_loader) // CONFIG["grad_accum_steps"]
total_steps = steps_per_epoch * CONFIG["num_epochs"]
warmup_steps = int(total_steps * CONFIG["warmup_ratio"])

optimizer = AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=0.01,
    fused=torch.cuda.is_available(),
)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Total steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"Precision: {'BF16' if CONFIG['use_bf16'] else 'FP16'}")

Steps per epoch: 1018
Total steps: 5090
Warmup steps: 509
Precision: BF16


## 10. Training Loop

Main training loop that iterates over epochs, trains the model, evaluates on the dev set, tracks metrics (F1, MCC, combined score), saves the best model, and creates checkpoints after each epoch. Supports resuming from a previous checkpoint.

In [15]:
# Initialize tracking
history = {"train_loss": [], "dev_loss": [], "dev_f1": [], "dev_acc": [], "dev_mcc": []}
best_f1 = 0
best_mcc = 0
best_combined = 0
start_epoch = 0

# Resume from checkpoint if specified
if CONFIG["resume_from_checkpoint"]:
    print(f"\nResuming from checkpoint: {CONFIG['resume_from_checkpoint']}")
    checkpoint = load_checkpoint(
        CONFIG["resume_from_checkpoint"], optimizer, scheduler, device
    )
    model = checkpoint["model"]
    start_epoch = checkpoint["epoch"] + 1
    best_f1 = checkpoint["best_metrics"]["f1"]
    best_mcc = checkpoint["best_metrics"]["mcc"]
    best_combined = checkpoint["best_metrics"]["combined"]
    history = checkpoint["history"]
    print(f"Resumed from epoch {start_epoch}, best F1: {best_f1:.4f}, best MCC: {best_mcc:.4f}")

In [16]:
# Training loop
print("=" * 60)
print("Starting Training")
print("=" * 60)

for epoch in range(start_epoch, CONFIG["num_epochs"]):
    print(f"\nEpoch {epoch+1}/{CONFIG['num_epochs']}")
    print("-" * 40)

    # Train
    train_loss = train_epoch(
        model, train_loader, optimizer, scheduler,
        device, CONFIG["grad_accum_steps"], CONFIG["use_bf16"]
    )

    # Evaluate
    metrics = evaluate(model, dev_loader, device, CONFIG["use_bf16"])

    # Combined score
    mcc_normalized = (metrics['mcc'] + 1) / 2
    combined_score = (metrics['f1'] + mcc_normalized) / 2

    # Log
    print(f"\nTrain Loss: {train_loss:.4f}")
    print(f"Dev Loss: {metrics['loss']:.4f}")
    print(f"Dev F1: {metrics['f1']:.4f}")
    print(f"Dev Accuracy: {metrics['accuracy']:.4f}")
    print(f"Dev MCC: {metrics['mcc']:.4f}")
    print(f"Combined Score: {combined_score:.4f}")

    if torch.cuda.is_available():
        print(f"GPU Memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
        torch.cuda.reset_peak_memory_stats()

    # History
    history["train_loss"].append(train_loss)
    history["dev_loss"].append(metrics["loss"])
    history["dev_f1"].append(metrics["f1"])
    history["dev_acc"].append(metrics["accuracy"])
    history["dev_mcc"].append(metrics["mcc"])

    # Save best model
    if combined_score > best_combined:
        best_combined = combined_score
        best_f1 = metrics["f1"]
        best_mcc = metrics["mcc"]
        
        model_to_save = model._orig_mod if hasattr(model, "_orig_mod") else model
        model_to_save.save(CONFIG["output_dir"])
        tokenizer.save_pretrained(CONFIG["output_dir"])
        print(f"*** New best! F1: {best_f1:.4f}, MCC: {best_mcc:.4f} - Model saved! ***")

    # Save checkpoint
    checkpoint_path = os.path.join(CONFIG["checkpoint_dir"], f"epoch_{epoch+1}")
    save_checkpoint(
        model, optimizer, scheduler, epoch,
        {"f1": best_f1, "mcc": best_mcc, "combined": best_combined},
        history, checkpoint_path
    )

    # Memory cleanup
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 60)
print(f"Training Complete!")
print(f"Best F1: {best_f1:.4f}")
print(f"Best MCC: {best_mcc:.4f}")
print(f"Best Combined Score: {best_combined:.4f}")
print(f"Model saved to: {CONFIG['output_dir']}")
print("=" * 60)

Starting Training

Epoch 1/5
----------------------------------------


Training:   0%|          | 0/6108 [00:00<?, ?it/s]

/tmp/ipykernel_406872/1282568413.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=autocast_dtype):
/tmp/ipykernel_406872/1282568413.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=autocast_dtype):


Evaluating:   0%|          | 0/842 [00:00<?, ?it/s]

/tmp/ipykernel_406872/1282568413.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=autocast_dtype):
/tmp/ipykernel_406872/1282568413.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=autocast_dtype):


...

Training:   0%|          | 0/6108 [00:00<?, ?it/s]

/tmp/ipykernel_406872/1282568413.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=autocast_dtype):


Evaluating:   0%|          | 0/842 [00:00<?, ?it/s]

/tmp/ipykernel_406872/1282568413.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=autocast_dtype):



Train Loss: 0.0930
Dev Loss: 0.2716
Dev F1: 0.9302
Dev Accuracy: 0.9277
Dev MCC: 0.8552
Combined Score: 0.9289
GPU Memory: 15.06 GB
Model saved to run6/rwkv7_checkpoints/epoch_5/model
Checkpoint saved to run6/rwkv7_checkpoints/epoch_5

Training Complete!
Best F1: 0.9303
Best MCC: 0.8555
Best Combined Score: 0.9290
Model saved to: run6/rwkv7_nli_model


## 11. Results Visualization

Plot training curves showing loss, F1 score, accuracy, and MCC across epochs. Saves the figure to the output directory.

In [ ]:
# Plot training history
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Loss curves (train vs dev)
axes[0].plot(epochs, history["train_loss"], 'b-o', label="Train")
axes[0].plot(epochs, history["dev_loss"], 'r-o', label="Dev")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(True)
axes[0].set_xticks(epochs)

# F1 score progression
axes[1].plot(epochs, history["dev_f1"], 'g-o')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1")
axes[1].set_title(f"Dev F1 (Best: {best_f1:.4f})")
axes[1].grid(True)
axes[1].set_xticks(epochs)

# Accuracy and MCC comparison
axes[2].plot(epochs, history["dev_acc"], 'b-o', label="Accuracy")
axes[2].plot(epochs, history["dev_mcc"], 'orange', marker='o', label="MCC")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Score")
axes[2].set_title("Dev Metrics")
axes[2].legend()
axes[2].grid(True)
axes[2].set_xticks(epochs)

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/training_curves.png", dpi=150)
plt.show()

In [18]:
# Final classification report
dev_loader = DataLoader(
    dev_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False,
)
print("\nFinal Classification Report:")
final = evaluate(model, dev_loader, device, CONFIG["use_bf16"])
print(classification_report(
    final["labels"], final["preds"],
    target_names=["Not Entailed", "Entailed"]
))


Final Classification Report:


Evaluating:   0%|          | 0/1684 [00:00<?, ?it/s]

/tmp/ipykernel_406872/1282568413.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=autocast_dtype):


              precision    recall  f1-score   support

Not Entailed       0.93      0.92      0.93      3258
    Entailed       0.93      0.93      0.93      3478

    accuracy                           0.93      6736
   macro avg       0.93      0.93      0.93      6736
weighted avg       0.93      0.93      0.93      6736



## 12. Generate Test Predictions

Load the test set (when available), generate predictions using the trained model, and save results to a CSV file. The output contains only prediction labels (0 or 1), one per line, matching the order of samples in the test set.

In [ ]:
# Generate predictions for test set (when available)
TEST_PATH = "Data/test.csv"  # Update this path

if os.path.exists(TEST_PATH):
    print("Loading test data...")
    test_dataset = NLIDataset(TEST_PATH, tokenizer, CONFIG["max_length"], is_test=True)
    test_loader = DataLoader(
        test_dataset,
        batch_size=CONFIG["batch_size"] * 2,
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
    )
    
    print("Generating predictions...")
    model.eval()
    all_preds = []
    
    autocast_dtype = torch.bfloat16 if CONFIG["use_bf16"] else torch.float16
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predicting"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            with torch.cuda.amp.autocast(dtype=autocast_dtype):
                out = model(input_ids, attention_mask)
            
            preds = torch.argmax(out["logits"], dim=1)
            all_preds.extend(preds.cpu().tolist())
    
    # Save predictions - generate id from row index since test.csv has no id column
    output_path = f"{CONFIG['output_dir']}/test_predictions.csv"
    output_df = pd.DataFrame({"prediction": all_preds})
    output_df.to_csv(output_path, index=False, header=False)
    print(f"Predictions saved to {output_path}")
else:
    print(f"Test file not found at {TEST_PATH}")
    print("Update TEST_PATH when test data is released.")

In [ ]:
# Save test predictions for submission
TEST_PATH = "Data/test.csv"
OUTPUT_FILENAME = "Group_CW2_B.csv"

test_df = pd.read_csv(TEST_PATH)
test_dataset = NLIDataset(TEST_PATH, tokenizer, CONFIG["max_length"], is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"] * 2, shuffle=False, num_workers=0)

model.eval()
all_preds = []
autocast_dtype = torch.bfloat16 if CONFIG["use_bf16"] else torch.float16

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        with torch.amp.autocast('cuda', dtype=autocast_dtype):
            out = model(input_ids, attention_mask)
        preds = torch.argmax(out["logits"], dim=1)
        all_preds.extend(preds.cpu().tolist())

# Save predictions only (no id column in test.csv, submission format is just predictions)
output_df = pd.DataFrame({"prediction": all_preds})
output_df.to_csv(OUTPUT_FILENAME, index=False, header=False)
print(f"Saved {len(all_preds)} predictions to {OUTPUT_FILENAME}")

## 13. Standalone Evaluation (No Training Required)

This cell runs independently without executing any previous cells. To evaluate the model:

1. Ensure the saved model directory exists (containing `backbone/`, `classifier.pt`, `config.pt`)
2. Update `model_path` and `eval_data_path` in `EVAL_CONFIG` to point to your files
3. Run only this cell

In [ ]:
# Standalone Evaluation Cell - Imports
import os
import json
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef, classification_report
from tqdm.auto import tqdm

# Configuration - update paths as needed
EVAL_CONFIG = {
    "model_path": "rwkv",
    "eval_data_path": "Data/dev.csv",
    "max_length": 320,
    "batch_size": 6,
    "use_bf16": True,
}

def get_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        if not torch.cuda.is_bf16_supported():
            print("BF16 not supported, using FP16")
            EVAL_CONFIG["use_bf16"] = False
        return device
    else:
        print("Using CPU")
        EVAL_CONFIG["use_bf16"] = False
        return torch.device("cpu")


class NLIDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_length=256, is_test=False):
        self.df = pd.read_csv(csv_path)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_test = is_test
        print(f"Loaded {len(self.df)} samples from {csv_path}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = f"Premise: {row['premise']}\nHypothesis: {row['hypothesis']}"
        enc = self.tokenizer(
            text, max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt"
        )
        item = {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
        }
        if not self.is_test and "label" in row:
            item["labels"] = torch.tensor(int(row["label"]), dtype=torch.long)
        return item


class RWKV7ForNLI(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.backbone(
            input_ids=input_ids, output_hidden_states=True, return_dict=True,
        )
        hidden = outputs.hidden_states[-1]
        if attention_mask is not None:
            seq_lens = attention_mask.sum(dim=1) - 1
            seq_lens = seq_lens.clamp(min=0)
            batch_idx = torch.arange(hidden.size(0), device=hidden.device)
            pooled = hidden[batch_idx, seq_lens]
        else:
            pooled = hidden[:, -1, :]
        pooled = pooled.float()
        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return {"loss": loss, "logits": logits}

    @classmethod
    def load(cls, path, device, use_bf16=True):
        """Load a saved model with LoRA adapters."""
        config = torch.load(os.path.join(path, "config.pt"), weights_only=False)
        model = cls.__new__(cls)
        nn.Module.__init__(model)

        dtype = torch.bfloat16 if use_bf16 else torch.float16
        backbone_path = os.path.join(path, "backbone")
        
        # Check if this is a PEFT model by looking for adapter_config.json
        adapter_config_path = os.path.join(backbone_path, "adapter_config.json")
        
        if os.path.exists(adapter_config_path):
            # Load as PEFT model: first load base, then load adapters
            with open(adapter_config_path, 'r') as f:
                adapter_config = json.load(f)
            base_model_name = adapter_config.get("base_model_name_or_path", "fla-hub/rwkv7-7.2B-g0a")
            
            print(f"Loading base model: {base_model_name}")
            base_model = AutoModelForCausalLM.from_pretrained(
                base_model_name,
                torch_dtype=dtype,
                trust_remote_code=True,
            )
            
            print(f"Loading LoRA adapters from: {backbone_path}")
            model.backbone = PeftModel.from_pretrained(base_model, backbone_path)
        else:
            # Fallback: load as regular model (no LoRA)
            print(f"Loading model from: {backbone_path}")
            model.backbone = AutoModelForCausalLM.from_pretrained(
                backbone_path,
                torch_dtype=dtype,
                trust_remote_code=True,
            )
        
        model.hidden_size = config["hidden_size"]
        model.dtype = dtype

        model.classifier = nn.Sequential(
            nn.LayerNorm(model.hidden_size),
            nn.Dropout(0.1),
            nn.Linear(model.hidden_size, 2)
        )
        model.classifier.load_state_dict(
            torch.load(os.path.join(path, "classifier.pt"), weights_only=False)
        )

        print("Model loaded successfully!")
        return model.to(device)


@torch.no_grad()
def evaluate(model, loader, device, use_bf16):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    autocast_dtype = torch.bfloat16 if use_bf16 else torch.float16
    for batch in tqdm(loader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        with torch.amp.autocast('cuda', dtype=autocast_dtype):
            out = model(input_ids, attention_mask, labels)
        preds = torch.argmax(out["logits"], dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        total_loss += out["loss"].item()
    return {
        "loss": total_loss / len(loader),
        "f1": f1_score(all_labels, all_preds),
        "accuracy": accuracy_score(all_labels, all_preds),
        "mcc": matthews_corrcoef(all_labels, all_preds),
        "preds": all_preds,
        "labels": all_labels,
    }


# Run evaluation
print("=" * 60)
print("RWKV-7 NLI Model Evaluation")
print("=" * 60)

device = get_device()

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(EVAL_CONFIG["model_path"], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("\nLoading model...")
model = RWKV7ForNLI.load(EVAL_CONFIG["model_path"], device, EVAL_CONFIG["use_bf16"])
model.eval()

print("\nLoading evaluation data...")
eval_dataset = NLIDataset(EVAL_CONFIG["eval_data_path"], tokenizer, EVAL_CONFIG["max_length"])
eval_loader = DataLoader(eval_dataset, batch_size=EVAL_CONFIG["batch_size"], shuffle=False, num_workers=0)

print("\nRunning evaluation...")
metrics = evaluate(model, eval_loader, device, EVAL_CONFIG["use_bf16"])

print("\n" + "=" * 60)
print("RESULTS")
print("=" * 60)
print(f"Loss:     {metrics['loss']:.4f}")
print(f"F1 Score: {metrics['f1']:.4f}")
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"MCC:      {metrics['mcc']:.4f}")
combined = (metrics['f1'] + (metrics['mcc'] + 1) / 2) / 2
print(f"Combined: {combined:.4f}")
print("=" * 60)

print("\nClassification Report:")
print(classification_report(metrics["labels"], metrics["preds"], target_names=["Not Entailed", "Entailed"]))